# Phase 1 - NBA Game Predictor

Exploration and plots for the Phase 1 logistic-regression model. All feature
engineering, the leakage guard, and the time-based split live in
`src/phase1/game_predictor.py`; this notebook reuses those functions so the
notebook and the script never disagree.

Contents:
1. Home win rate by rest-day advantage
2. Calibration curve (are predicted probabilities honest?)
3. Model coefficients
4. Home-court advantage over time — a COVID natural experiment

In [ ]:
import sys
from pathlib import Path

# Make the project root importable so `src...` works from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

from src.ingest.nba_data import DATA_DIR, recent_seasons
from src.phase1.game_predictor import (
    build_game_table, time_split, evaluate,
    BASE_FEATURES, EXTRA_FEATURES, N_SEASONS, LAST_SEASON_START, N_TEST_SEASONS,
)

sns.set_theme(style='whitegrid')

# Load the processed features cached by the script; build them if missing.
feat_path = DATA_DIR / 'phase1_game_features.csv'
if feat_path.exists():
    games = pd.read_csv(feat_path)
else:
    games = build_game_table(recent_seasons(N_SEASONS, LAST_SEASON_START))
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])
print(games.shape)
games.head()

## 1. Home win rate by rest-day advantage

`d_rest` is home rest days minus away rest days. If rest matters, teams with a
rest advantage should win more often. This is a raw data check, not the model.

In [ ]:
def rest_bucket(d):
    if d <= -2: return '<= -2 (away rested)'
    if d == -1: return '-1'
    if d == 0:  return '0 (even)'
    if d == 1:  return '+1'
    return '>= +2 (home rested)'

order = ['<= -2 (away rested)', '-1', '0 (even)', '+1', '>= +2 (home rested)']
tmp = games.assign(rest_adv=games['d_rest'].apply(rest_bucket))
rate = tmp.groupby('rest_adv')['target'].mean().reindex(order)

plt.figure(figsize=(8, 4.5))
ax = sns.barplot(x=rate.index, y=rate.values, color='steelblue')
ax.axhline(games['target'].mean(), ls='--', color='gray',
           label=f"overall home win rate = {games['target'].mean():.3f}")
ax.set(xlabel='Home rest-day advantage', ylabel='Home win rate',
       title='Home win rate rises with rest advantage')
plt.xticks(rotation=15); plt.legend(); plt.tight_layout(); plt.show()

## 2. Calibration curve

We refit the extended model on the training seasons and check, on the held-out
test seasons, whether predicted probabilities match observed win rates. A model
on the diagonal is well calibrated.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train, test, test_seasons = time_split(games, N_TEST_SEASONS)
feats = BASE_FEATURES + EXTRA_FEATURES
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(train[feats], train['target'])
prob = model.predict_proba(test[feats])[:, 1]

frac_pos, mean_pred = calibration_curve(test['target'], prob, n_bins=10)
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], ls='--', color='gray', label='perfectly calibrated')
plt.plot(mean_pred, frac_pos, marker='o', color='crimson', label='model')
plt.xlabel('Mean predicted P(home win)'); plt.ylabel('Observed home win rate')
plt.title(f'Calibration on test seasons {test_seasons}')
plt.legend(); plt.tight_layout(); plt.show()

## 3. Model coefficients

Features are standardized, so coefficient magnitude is a fair importance
comparison. Sign shows direction of effect on P(home win).

In [ ]:
coefs = pd.Series(model.named_steps['logisticregression'].coef_[0],
                  index=feats).sort_values()
plt.figure(figsize=(8, 4.5))
colors = ['crimson' if v < 0 else 'steelblue' for v in coefs.values]
sns.barplot(x=coefs.values, y=coefs.index, hue=coefs.index,
            palette=colors, legend=False)
plt.axvline(0, color='gray', lw=1)
plt.xlabel('Standardized logistic-regression coefficient')
plt.title('What drives the home-win prediction'); plt.tight_layout(); plt.show()

# Print the headline metrics alongside the baseline for the README.
_ = evaluate(train, test, feats, 'Extended model (notebook)')

## Takeaways

- The model beats the naive home-wins baseline on both accuracy and log loss.
- Rolling scoring margin (`d_pts`, `d_pts_allowed`) does most of the work;
  rest and back-to-back effects are real but small.
- Accuracy stays in the mid-60s % - consistent with public-data NBA prediction
  and well below the ~70% level that would suggest leakage.
- **Crowd effect:** home win% drops from ~58.6% (2015-19) to ~54% in the empty-
  arena COVID seasons, then never fully rebounds — the crowd contributes to home
  advantage, but a broader league-wide decline is also underway.

In [ ]:
# Home-court advantage over time -- a COVID natural experiment.
from src.ingest.nba_data import recent_seasons, team_game_logs

logs = team_game_logs(recent_seasons(10, 2024))
logs['is_home'] = logs['MATCHUP'].str.contains('vs.', regex=False)
logs['win'] = (logs['WL'] == 'W').astype(int)

def season_label(sid):
    y = int(str(sid)[1:]); return f'{y}-{str(y + 1)[-2:]}'

hc = (logs[logs['is_home']].groupby('SEASON_ID')['win'].mean()
      .rename(index=season_label).rename('home_win_pct'))
covid = {'2019-20', '2020-21'}  # Orlando bubble + reduced-capacity arenas
colors = ['crimson' if s in covid else 'steelblue' for s in hc.index]
pre_covid = hc.loc[['2015-16', '2016-17', '2017-18', '2018-19']].mean()

plt.figure(figsize=(9, 4.5))
sns.barplot(x=hc.index, y=hc.values, hue=hc.index, palette=colors, legend=False)
plt.axhline(pre_covid, ls='--', color='gray', label=f'pre-COVID avg = {pre_covid:.3f}')
plt.axhline(0.5, ls=':', color='black', lw=1, label='no home advantage')
plt.ylim(0.5, 0.62); plt.ylabel('Home win %'); plt.xlabel('Season')
plt.title('Home-court advantage dips in the empty-arena seasons (red)')
plt.xticks(rotation=30); plt.legend(); plt.tight_layout(); plt.show()
hc.round(3)

## Takeaways

- The model beats the naive home-wins baseline on both accuracy and log loss.
- Rolling scoring margin (`d_pts`, `d_pts_allowed`) does most of the work;
  rest and back-to-back effects are real but small.
- Accuracy stays in the mid-60s % - consistent with public-data NBA prediction
  and well below the ~70% level that would suggest leakage.